# Response Clarity Classification: Transformer Fine-Tuning

**Student ID:** sdi2200160
**Course:** Artificial Intelligence II - Deep Learning for NLP
**Assignment:** Homework 2 - Transformer Fine-Tuning
**Due Date:** April 24, 2026

---

## Overview

Fine-tune a pretrained transformer model for response clarity classification
on political question-answer pairs (CLARITY dataset).

**Architecture:** Same protocol-based `Classifier` pipeline as HW1 -- only the
source encoder (`TokenizerEncoder`) and model (`TransformerModel`) are swapped in.

**Input construction:** Sentence pair encoding `[CLS] question [SEP] answer [SEP]`

**Training:** Custom training loop (no HF Trainer API) with AdamW, linear warmup + decay,
gradient clipping, class-weighted loss, and best-model checkpointing.

In [ ]:
from __future__ import annotations

# Standard library
import functools
import os
import random
import typing

# Required for DeBERTa v3 tokenizer (SentencePiece -> protobuf)
os.environ.setdefault('PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION', 'python')

# Data science
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch
import torch
import torch.nn
import torch.utils.data

# HuggingFace
import transformers
import datasets

# sklearn
import sklearn.base
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing

# Reproducibility
RANDOM_STATE = 42

# Visualization
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
	print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Configuration

In [ ]:
def seed_everything(seed: int = RANDOM_STATE) -> None:
	"""Set all random seeds for reproducibility."""
	random.seed(seed)
	np.random.seed(seed)
	torch.manual_seed(seed)
	torch.cuda.manual_seed_all(seed)
	torch.backends.cudnn.deterministic = True
	torch.backends.cudnn.benchmark = False

seed_everything()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ========== CHANGE THIS PER NOTEBOOK ==========
MODEL_NAME = 'bert-base-uncased'
# MODEL_NAME = 'distilbert-base-uncased'
# MODEL_NAME = 'microsoft/deberta-v3-base'
# ===============================================

print(f'Device: {DEVICE}')
print(f'Model:  {MODEL_NAME}')

## 2. Pipeline Framework

We reuse the same protocol-based `Classifier` pipeline from HW1 -- the exact same
orchestration that ran TF-IDF + Logistic Regression now runs Transformer fine-tuning.
We only swap in two new components:

| HW1 | HW2 |
|-----|-----|
| `TfidfVectorizer` | `TokenizerEncoder` (wraps `AutoTokenizer`) |
| `LogisticRegression` | `TransformerModel` (custom training loop) |
| `ChainPreprocessor(CleanText, Lemmatize)` | `IdentityPreprocessor` (tokenizer handles text) |
| `question + " \| " + answer` (Series) | `DataFrame{question, answer}` (pair encoding) |

Everything else is inherited: `Classifier`, `LabelEncoder`, scorers, `compile()`, `score()`.

In [ ]:
# Protocol definitions (same as HW1 / shared codebase)
Float = float | np.float16 | np.float32

@typing.runtime_checkable
class Preprocessor[Decoded](typing.Protocol):
	def __call__(self, source: Decoded) -> Decoded: ...

@typing.runtime_checkable
class Scorer[Target, Result](typing.Protocol):
	def __call__(self, true: Target, pred: Target, /) -> Result: ...

@typing.runtime_checkable
class Encoder[Decoded, Encoded](typing.Protocol):
	def fit(self, source: Decoded, signal: typing.Any | None = None, /) -> typing.Self: ...
	def transform(self, source: Decoded, /) -> Encoded: ...

@typing.runtime_checkable
class Bicoder[Decoded, Encoded](Encoder[Decoded, Encoded], typing.Protocol):
	def inverse_transform(self, target: Encoded, /) -> Decoded: ...

@typing.runtime_checkable
class Model[Source, Target](typing.Protocol):
	def fit(self, source: Source, target: Target, /) -> typing.Self: ...
	def predict(self, source: Source, /) -> Target: ...

class IdentityPreprocessor[Decoded]:
	def __call__(self, source: Decoded) -> Decoded:
		return source

print('Protocols defined')

In [ ]:
class Classifier[DecodedSource, EncodedSource, EncodedTarget, DecodedTarget](
	sklearn.base.BaseEstimator,
	sklearn.base.ClassifierMixin,
):
	"""Encoder-agnostic classification pipeline (same as HW1)."""

	def __init__(self,
		preprocessor: Preprocessor[DecodedSource],
		model: Model[EncodedSource, EncodedTarget],
		source_encoder: Encoder[DecodedSource, EncodedSource],
		target_bicoder: Bicoder[DecodedTarget, EncodedTarget],
		scorers: dict[str, Scorer[EncodedTarget, Float]] | None = None,
	) -> None:
		self.preprocessor = preprocessor
		self.model = model
		self.source_encoder = source_encoder
		self.target_bicoder = target_bicoder
		self.scorers = scorers or {}

	def compile(self, **scorers) -> typing.Self:
		self.scorers = scorers
		return self

	def preprocess(self, source):
		return self.preprocessor(source)

	def fit(self, source, target, /):
		source = self.preprocess(source)
		self.source_encoder.fit(source, target)
		self.target_bicoder.fit(target)
		self.model.fit(
			self.source_encoder.transform(source),
			self.target_bicoder.transform(target),
		)
		return self

	def forward(self, source):
		return self.model.predict(
			self.source_encoder.transform(self.preprocess(source))
		)

	def predict(self, source):
		return self.target_bicoder.inverse_transform(
			self.forward(self.preprocess(source))
		)

	def score(self, source, target, /):
		true = self.target_bicoder.transform(target)
		pred = self.forward(source)
		return {name: scorer(true, pred) for name, scorer in self.scorers.items()}

print('Classifier pipeline defined')

## 3. Data Loading

In [ ]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
	from kaggle_secrets import UserSecretsClient
	os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
else:
	import dotenv
	dotenv.load_dotenv(override=True)

data = datasets.load_dataset('ailsntua/QEvasion').select_columns([
	'question', 'interview_answer', 'clarity_label',
])

train_df = data['train'].to_pandas()
test_df = data['test'].to_pandas()

print(f'Train: {len(train_df)} samples')
print(f'Test:  {len(test_df)} samples')
train_df.head()

## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, df) in zip(axes, [('Train', train_df), ('Test', test_df)]):
	counts = df['clarity_label'].value_counts()
	ax.bar(counts.index, counts.values, color=['#2ecc71', '#e74c3c', '#3498db'])
	ax.set_title(f'{name} Set Class Distribution', fontsize=14, fontweight='bold')
	ax.set_xlabel('Clarity Label')
	ax.set_ylabel('Count')
	ax.grid(axis='y', alpha=0.3)
	for i, (label, count) in enumerate(counts.items()):
		ax.text(i, count + max(counts) * 0.02,
			f'{count}
({count/len(df)*100:.1f}%)', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
train_df['q_len'] = train_df['question'].fillna('').str.split().str.len()
train_df['a_len'] = train_df['interview_answer'].fillna('').str.split().str.len()
train_df['total_len'] = train_df['q_len'] + train_df['a_len']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col, title in zip(axes,
	['q_len', 'a_len', 'total_len'],
	['Question Length', 'Answer Length', 'Total Length'],
):
	train_df.boxplot(column=col, by='clarity_label', ax=ax)
	ax.set_title(f'{title} by Class', fontsize=12, fontweight='bold')
	ax.set_xlabel('Clarity Label')
	ax.set_ylabel('Word Count')

plt.suptitle('')
plt.tight_layout()
plt.show()

## 5. New Components

These are the only two new classes needed. Everything else is reused from HW1.

### ClarityDataset

Internal PyTorch Dataset for the training loop.

In [ ]:
class ClarityDataset(torch.utils.data.Dataset):
	def __init__(self, encodings, labels=None):
		self.encodings = encodings
		self.labels = labels

	def __len__(self):
		return len(self.encodings['input_ids'])

	def __getitem__(self, idx):
		item = {key: val[idx] for key, val in self.encodings.items()}
		if self.labels is not None:
			item['labels'] = self.labels[idx]
		return item

print('ClarityDataset defined')

### TokenizerEncoder (Encoder protocol)

Replaces TF-IDF / Word2Vec. Wraps `AutoTokenizer` with sentence pair encoding.
`fit()` is a no-op (pretrained). `transform()` tokenizes a DataFrame of Q-A pairs.

In [ ]:
class TokenizerEncoder(sklearn.base.BaseEstimator, sklearn.base.TransformerMixin):
	"""Wraps AutoTokenizer as an Encoder[DataFrame, BatchEncoding]."""

	def __init__(self, model_name='bert-base-uncased', max_length=256):
		self.model_name = model_name
		self.max_length = max_length

	def fit(self, source, signal=None):
		return self  # pretrained, no fitting needed

	def transform(self, source):
		if not hasattr(self, '_tokenizer') or self._tokenizer is None:
			self._tokenizer = transformers.AutoTokenizer.from_pretrained(self.model_name)
		return self._tokenizer(
			source['question'].tolist(), source['answer'].tolist(),
			padding=True, truncation=True,
			max_length=self.max_length, return_tensors='pt',
		)

print('TokenizerEncoder defined')

### TransformerModel (Model protocol)

Replaces LogisticRegression. Custom training loop inside `fit()`, batch inference in `predict()`.

Training loop: AdamW (no decay on bias/LayerNorm), linear warmup + decay,
gradient clipping, class-weighted loss, internal val split with best-model checkpointing.

In [ ]:
class TransformerModel(sklearn.base.BaseEstimator):
	"""Wraps transformer fine-tuning as a Model[BatchEncoding, ndarray]."""

	def __init__(self, model_name='bert-base-uncased', num_labels=3,
		batch_size=16, learning_rate=2e-5, weight_decay=0.01,
		num_epochs=4, warmup_ratio=0.1, max_grad_norm=1.0,
		class_weights=None, val_frac=0.15, device=DEVICE,
	):
		self.model_name = model_name
		self.num_labels = num_labels
		self.batch_size = batch_size
		self.learning_rate = learning_rate
		self.weight_decay = weight_decay
		self.num_epochs = num_epochs
		self.warmup_ratio = warmup_ratio
		self.max_grad_norm = max_grad_norm
		self.class_weights = class_weights
		self.val_frac = val_frac
		self.device = device
		self._model = None
		self.history = {}

	def _build(self):
		self._model = transformers.AutoModelForSequenceClassification.from_pretrained(
			self.model_name, num_labels=self.num_labels,
		).to(self.device)

	def fit(self, source, target, /):
		if self._model is None:
			self._build()

		labels = torch.tensor(target, dtype=torch.long) if not isinstance(target, torch.Tensor) else target

		# Internal train/val split
		if self.val_frac > 0:
			idx = np.arange(len(labels))
			train_idx, val_idx = sklearn.model_selection.train_test_split(
				idx, test_size=self.val_frac, random_state=RANDOM_STATE, stratify=target,
			)
			train_enc = {k: v[train_idx] for k, v in source.items()}
			val_enc = {k: v[val_idx] for k, v in source.items()}
			train_labels, val_labels = labels[train_idx], labels[val_idx]
		else:
			train_enc, train_labels = dict(source), labels
			val_enc, val_labels = None, None

		train_ds = ClarityDataset(train_enc, train_labels)
		train_loader = torch.utils.data.DataLoader(train_ds, batch_size=self.batch_size, shuffle=True)

		# Optimizer
		no_decay = {'bias', 'LayerNorm.weight', 'LayerNorm.bias'}
		param_groups = [
			{'params': [p for n, p in self._model.named_parameters()
				if not any(nd in n for nd in no_decay)], 'weight_decay': self.weight_decay},
			{'params': [p for n, p in self._model.named_parameters()
				if any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
		]
		optimizer = torch.optim.AdamW(param_groups, lr=self.learning_rate)

		total_steps = len(train_loader) * self.num_epochs
		scheduler = transformers.get_linear_schedule_with_warmup(
			optimizer, int(total_steps * self.warmup_ratio), total_steps,
		)

		loss_fn = torch.nn.CrossEntropyLoss(
			weight=self.class_weights.to(self.device) if self.class_weights is not None else None,
		)

		has_val = val_enc is not None and val_labels is not None
		self.history = {'train_loss': [], 'val_loss': [], 'val_f1': []}
		best_val_f1, best_state = 0.0, None

		for epoch in range(self.num_epochs):
			self._model.train()
			total_loss = 0.0
			for batch in train_loader:
				optimizer.zero_grad()
				model_inputs = {k: v.to(self.device) for k, v in batch.items() if k != 'labels'}
				outputs = self._model(**model_inputs)
				loss = loss_fn(outputs.logits, batch['labels'].to(self.device))
				loss.backward()
				torch.nn.utils.clip_grad_norm_(self._model.parameters(), self.max_grad_norm)
				optimizer.step()
				scheduler.step()
				total_loss += loss.item()

			avg_loss = total_loss / len(train_loader)
			self.history['train_loss'].append(avg_loss)

			if has_val:
				val_metrics = self._evaluate(val_enc, val_labels)
				self.history['val_loss'].append(val_metrics['loss'])
				self.history['val_f1'].append(val_metrics['f1'])
				improved = val_metrics['f1'] > best_val_f1
				if improved:
					best_val_f1 = val_metrics['f1']
					best_state = {k: v.cpu().clone() for k, v in self._model.state_dict().items()}
				print(f'Epoch {epoch+1}/{self.num_epochs} | '
					f'Train Loss: {avg_loss:.4f} | '
					f'Val Loss: {val_metrics["loss"]:.4f} | '
					f'Val F1: {val_metrics["f1"]:.4f}'
					f'{"  *" if improved else ""}')
			else:
				print(f'Epoch {epoch+1}/{self.num_epochs} | Train Loss: {avg_loss:.4f}')

		if best_state is not None:
			self._model.load_state_dict(best_state)
			self._model.to(self.device)
			print(f'
Restored best model (Val F1: {best_val_f1:.4f})')
		return self

	@torch.no_grad()
	def _evaluate(self, encodings, labels):
		self._model.eval()
		loader = torch.utils.data.DataLoader(
			ClarityDataset(encodings, labels), batch_size=self.batch_size,
		)
		all_preds, total_loss = [], 0.0
		loss_fn = torch.nn.CrossEntropyLoss()
		for batch in loader:
			model_inputs = {k: v.to(self.device) for k, v in batch.items() if k != 'labels'}
			outputs = self._model(**model_inputs)
			total_loss += loss_fn(outputs.logits, batch['labels'].to(self.device)).item()
			all_preds.extend(outputs.logits.argmax(dim=-1).cpu().tolist())
		preds, true = np.array(all_preds), labels.numpy()
		return {
			'loss': total_loss / len(loader),
			'f1': sklearn.metrics.f1_score(true, preds, average='macro', zero_division=0),
		}

	@torch.no_grad()
	def predict(self, source, /):
		self._model.eval()
		loader = torch.utils.data.DataLoader(
			ClarityDataset(source), batch_size=self.batch_size,
		)
		all_preds = []
		for batch in loader:
			model_inputs = {k: v.to(self.device) for k, v in batch.items()}
			outputs = self._model(**model_inputs)
			all_preds.extend(outputs.logits.argmax(dim=-1).cpu().tolist())
		return np.array(all_preds)

print('TransformerModel defined')

## 6. Data Preparation

In [ ]:
def compute_class_weights(labels, num_classes=3):
	"""Balanced class weights: w_c = n / (k * count_c)."""
	counts = torch.bincount(labels, minlength=num_classes).float()
	return len(labels) / (num_classes * counts)

def macro_averaged(metric_fn, **kwargs):
	return functools.partial(metric_fn, average='macro', zero_division=0, **kwargs)

# Prepare data as DataFrames (question + answer columns)
X_train = pd.DataFrame({
	'question': train_df.question.fillna('').str.strip(),
	'answer': train_df.interview_answer.fillna('').str.strip(),
})
y_train = train_df.clarity_label.fillna('').str.strip()

X_test = pd.DataFrame({
	'question': test_df.question.fillna('').str.strip(),
	'answer': test_df.interview_answer.fillna('').str.strip(),
})
y_test = test_df.clarity_label.fillna('').str.strip()

# Class weights
label_encoder = sklearn.preprocessing.LabelEncoder()
label_encoder.fit(y_train)
class_weights = compute_class_weights(
	torch.tensor(label_encoder.transform(y_train), dtype=torch.long),
)

print(f'Training: {len(X_train)} samples')
print(f'Test:     {len(X_test)} samples')
print(f'Labels:   {list(label_encoder.classes_)}')
print(f'Weights:  {dict(zip(label_encoder.classes_, class_weights.tolist()))}')

## 7. Training

Assemble the pipeline -- same \ as HW1, just different components plugged in.

In [ ]:
seed_everything()

# Build the same Classifier pipeline, swapping in transformer components
classifier = Classifier(
	preprocessor=IdentityPreprocessor(),
	model=TransformerModel(
		model_name=MODEL_NAME,
		class_weights=class_weights,
		num_epochs=4,
		batch_size=16,
		learning_rate=2e-5,
		max_grad_norm=1.0,
		val_frac=0.15,
	),
	source_encoder=TokenizerEncoder(
		model_name=MODEL_NAME,
		max_length=256,
	),
	target_bicoder=label_encoder,
)

classifier.compile(
	accuracy=sklearn.metrics.accuracy_score,
	precision=macro_averaged(sklearn.metrics.precision_score),
	recall=macro_averaged(sklearn.metrics.recall_score),
	f1=macro_averaged(sklearn.metrics.f1_score),
)

print(f'Training {MODEL_NAME}...')
print(f'  Device: {DEVICE}')
print()

# Classifier.fit handles the full pipeline:
# IdentityPreprocessor -> TokenizerEncoder -> LabelEncoder -> TransformerModel
classifier.fit(X_train, y_train)

## 8. Evaluation

In [ ]:
# classifier.score() uses the same infrastructure as HW1
test_scores = classifier.score(X_test, y_test)

print('=' * 60)
print(f'TEST SET EVALUATION: {MODEL_NAME}')
print('=' * 60)
for name, score in test_scores.items():
	print(f'{name:12s} {score:.4f}')

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# classifier.predict() -> decoded string labels (via LabelEncoder)
y_pred = classifier.predict(X_test)

# For confusion matrix we need encoded labels
y_pred_enc = label_encoder.transform(y_pred)
y_true_enc = label_encoder.transform(y_test)

cm = confusion_matrix(y_true_enc, y_pred_enc)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title(f'{MODEL_NAME} - Confusion Matrix', fontsize=14, fontweight='bold')
ax.grid(False)
plt.tight_layout()
plt.show()

print('
Per-class classification report:')
print(sklearn.metrics.classification_report(
	y_true_enc, y_pred_enc,
	target_names=label_encoder.classes_,
	digits=4,
))

In [ ]:
model = classifier.model  # access the TransformerModel
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(model.history['train_loss']) + 1)

axes[0].plot(epochs, model.history['train_loss'], 'b-o', label='Train Loss')
if model.history['val_loss']:
	axes[0].plot(epochs, model.history['val_loss'], 'r-o', label='Val Loss')
axes[0].set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

if model.history['val_f1']:
	axes[1].plot(epochs, model.history['val_f1'], 'g-o', label='Val F1 (macro)')
	axes[1].set_title('Validation F1 Score', fontsize=14, fontweight='bold')
	axes[1].set_xlabel('Epoch')
	axes[1].set_ylabel('F1 Score')
	axes[1].legend()
	axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Submission

In [ ]:
submission = pd.Series(y_pred, name='Predicted')
submission.index.name = 'Id'

model_slug = MODEL_NAME.replace('/', '-')
submission_path = f'submission_{model_slug}.csv'
submission.to_csv(submission_path)

print(f'Saved: {submission_path}')
print(f'
Prediction distribution:')
print(submission.value_counts())
print(f'
Sample predictions:')
submission.head(10)

## 10. Conclusion

*TODO: Add analysis and discussion here.*

### Topics to Cover

- Effect of hyperparameter choices (learning rate, batch size, epochs, max length)
- Effect of input formulation (separator tokens, truncation strategy)
- Comparison with other models (see companion notebooks)
- Error analysis: which class is hardest, recurring patterns in errors
- What changes were attempted and whether they improved performance